# Runtime Environment Setup

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tarfile

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

REPO_ROOT_CANDIDATES = []
if IN_COLAB:
    REPO_ROOT_CANDIDATES.extend([
        Path("/content/560-FaceRecognition"),
        Path("/content/drive/MyDrive/560/Final-Project/560-FaceRecognition"),
        Path("/content/drive/MyDrive/560-FaceRecognition"),
    ])
else:
    cwd = Path.cwd()
    REPO_ROOT_CANDIDATES.extend([
        cwd,
        cwd / "560-FaceRecognition",
        Path.home() / "Desktop" / "560" / "Final-Project" / "560-FaceRecognition",
    ])

def has_repo_code(root: Path) -> bool:
    return (root / "project-fr" / "train_example.py").exists()

REPO_ROOT = next((path.resolve() for path in REPO_ROOT_CANDIDATES if has_repo_code(path)), None)

if REPO_ROOT is None and IN_COLAB:
    clone_target = Path("/content/560-FaceRecognition")
    if clone_target.exists():
        shutil.rmtree(clone_target)
    subprocess.run([
        "git",
        "clone",
        "https://github.com/ethan-yountz/560-FaceRecognition",
        str(clone_target),
    ], check=True)
    REPO_ROOT = clone_target.resolve()
elif REPO_ROOT is None:
    raise FileNotFoundError("Could not find 560-FaceRecognition with project-fr/train_example.py on this machine.")

PROJECT_DIR = REPO_ROOT / "project-fr"

DATASET_DIR_CANDIDATES = [PROJECT_DIR / "datasets" / "dataset_a"]
if IN_COLAB:
    DATASET_DIR_CANDIDATES.extend([
        Path("/content/drive/MyDrive/560/Final-Project/560-FaceRecognition/project-fr/datasets/dataset_a"),
        Path("/content/drive/MyDrive/560-FaceRecognition/project-fr/datasets/dataset_a"),
        Path("/content/drive/MyDrive/datasets/dataset_a"),
    ])

def has_dataset_files(path: Path) -> bool:
    return (path / "test.parquet").exists() and (path / "pairs.parquet").exists()

DATASET_DIR = next((path.resolve() for path in DATASET_DIR_CANDIDATES if has_dataset_files(path)), None)
if DATASET_DIR is None:
    searched = "\n".join(str(path) for path in DATASET_DIR_CANDIDATES)
    raise FileNotFoundError(
        "Could not find dataset_a.\n"
        "If you are using Colab, upload the dataset files into Google Drive at\n"
        "/content/drive/MyDrive/560/Final-Project/560-FaceRecognition/project-fr/datasets/dataset_a\n"
        f"Searched:\n{searched}"
    )

SPLIT_DIR = DATASET_DIR / "splits" / "val_15_seed42"

if IN_COLAB:
    COLAB_DATA_ROOT = Path("/content/comp560_data")
    WORK_DATASET_DIR = COLAB_DATA_ROOT / "dataset_a"
    WORK_DATASET_DIR.mkdir(parents=True, exist_ok=True)

    for filename in ("test.parquet", "pairs.parquet"):
        src = DATASET_DIR / filename
        dst = WORK_DATASET_DIR / filename
        if (not dst.exists()) or (src.stat().st_size != dst.stat().st_size):
            shutil.copy2(src, dst)

    drive_archive = DATASET_DIR / "images.tar.gz"
    local_archive = WORK_DATASET_DIR / "images.tar.gz"
    if drive_archive.exists() and ((not local_archive.exists()) or (drive_archive.stat().st_size != local_archive.stat().st_size)):
        shutil.copy2(drive_archive, local_archive)

    local_images_dir = WORK_DATASET_DIR / "images"
    needs_extract = (not local_images_dir.exists()) or (not any(local_images_dir.iterdir()))
    if needs_extract:
        if not local_archive.exists():
            raise FileNotFoundError(f"Missing images archive at {local_archive}")
        with tarfile.open(local_archive) as tar:
            tar.extractall(WORK_DATASET_DIR, filter="data")

    TRAIN_DATASET_DIR = WORK_DATASET_DIR
    IMAGES_DIR = local_images_dir
    SAVES_DIR = Path("/content/drive/MyDrive/560/Final-Project/560-FaceRecognition/project-fr/saves")
else:
    TRAIN_DATASET_DIR = DATASET_DIR
    IMAGES_ARCHIVE = DATASET_DIR / "images.tar.gz"
    IMAGES_DIR = DATASET_DIR / "images"
    needs_extract = (not IMAGES_DIR.exists()) or (not any(IMAGES_DIR.iterdir()))
    if needs_extract and IMAGES_ARCHIVE.exists():
        with tarfile.open(IMAGES_ARCHIVE) as tar:
            tar.extractall(DATASET_DIR, filter="data")
    SAVES_DIR = PROJECT_DIR / "saves"

SAVES_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

print(f"Colab: {IN_COLAB}")
print(f"Repo root: {REPO_ROOT}")
print(f"Project dir: {PROJECT_DIR}")
print(f"train_example.py exists: {(PROJECT_DIR / 'train_example.py').exists()}")
print(f"Drive dataset dir: {DATASET_DIR}")
print(f"Training dataset dir: {TRAIN_DATASET_DIR}")
print(f"Saves dir: {SAVES_DIR}")


In [ ]:
from pathlib import Path
print(f"Working directory: {Path.cwd()}")


In [ ]:
from pathlib import Path
cwd = Path.cwd()
print(cwd)
print("\n".join(sorted(path.name for path in cwd.iterdir())))


In [ ]:
required_files = [TRAIN_DATASET_DIR / "test.parquet", TRAIN_DATASET_DIR / "pairs.parquet"]
for path in required_files:
    print(f"{path.name}: {'found' if path.exists() else 'missing'}")
print(f"Validation split available on Drive: {SPLIT_DIR.exists()}")


In [ ]:
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"Images directory not found at {IMAGES_DIR}")

sample_entry = next(IMAGES_DIR.iterdir(), None)
print(f"Images directory ready: {IMAGES_DIR}")
print(f"Sample entry: {sample_entry.name if sample_entry else 'empty directory'}")


In [ ]:
from datetime import datetime

log_path = SAVES_DIR / "logs.txt"
with log_path.open("a", encoding="utf-8") as handle:
    handle.write(f"{datetime.now().isoformat()}\n")

print(f"Logging to {log_path}")


# Train Model

In [ ]:
import subprocess
import sys

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    device = "cpu"

cmd = [
    sys.executable,
    str(PROJECT_DIR / "train_example.py"),
    "--data_root", str(TRAIN_DATASET_DIR),
    "--save_dir", str(SAVES_DIR),
    "--device", device,
    "--num_workers", "2" if device == "cuda" else "0",
    "--batch_size", "64",
]

if SPLIT_DIR.exists():
    cmd.extend([
        "--train_metadata", str(SPLIT_DIR / "train_metadata.parquet"),
        "--val_metadata", str(SPLIT_DIR / "val_metadata.parquet"),
        "--val_pairs", str(SPLIT_DIR / "val_pairs.parquet"),
    ])

if device == "cuda":
    cmd.append("--amp")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True, cwd=str(PROJECT_DIR))
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Training command failed with exit code {result.returncode}")
